# Singular Value Decomposition — Step by Step

### Movie Ratings: 3 People × 2 Movies · **NumPy only**

This notebook works through the *exact same* example as the worksheet, one step at a
time, printing the result after every step.

| Name | Movie 1 | Movie 2 |
|:--|:--:|:--:|
| John | 5 | 2 |
| Mary | 4 | 4 |
| David | 2 | 5 |

**Goal:** factor the ratings matrix into

$$ A = U \Sigma V^{\mathsf T} $$

**The plan**

| Step | Do this |
|:--:|:--|
| 0 | Write the data as a matrix $A$ |
| 1 | Compute $A^{\mathsf T}$ |
| 2 | Compute $A^{\mathsf T}A$ and $AA^{\mathsf T}$ |
| 3 | Find the eigenvalues $\lambda_i$ |
| 4 | $\sigma_i = \sqrt{\lambda_i}$, build $\Sigma$ |
| 5 | Eigenvectors of $A^{\mathsf T}A$ → $V$ |
| 6 | $u_i = \frac{1}{\sigma_i} A v_i$ → $U$ |
| 7 | Assemble and **verify** |
| 8 | Interpret the result |
| 9 | Rank-1 approximation |
| 10 | Pseudoinverse |
| 11 | Cross-check against `np.linalg.svd` |

> The only import in this notebook is `numpy`. Every quantity is computed from
> scratch first, then checked against NumPy's built-in routine.

---
## Setup

One import, plus two small helpers to keep the printing readable.

In [1]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

names  = ["John", "Mary", "David"]
movies = ["Movie 1", "Movie 2"]


def show(label, M):
    """Print a labelled matrix or vector."""
    print(f"{label} =")
    print(np.round(np.asarray(M, dtype=float), 4))
    print()


def rule(title):
    """Print a section banner."""
    print("=" * 62)
    print(title)
    print("=" * 62)


print("numpy version:", np.__version__)
print("setup complete")

numpy version: 2.0.2
setup complete


---
## Step 0 — Write the data as a matrix

Rows are **people**, columns are **movies**. The size is *rows × columns*, so
$A$ is $3\times 2$: $m = 3$, $n = 2$.

> **Careful:** the math writes $a_{11}$ for the top-left entry (1-indexed), but
> NumPy writes `A[0, 0]` (0-indexed). So $a_{ij}$ is `A[i-1, j-1]`.

In [2]:
rule("STEP 0 - the data matrix A")

A = np.array([[5., 2.],     # John
              [4., 4.],     # Mary
              [2., 5.]])    # David

show("A", A)

m, n = A.shape
print("shape (m x n):", A.shape)
print(f"m = {m} people, n = {n} movies")
print()

# math index a_ij  ->  numpy index A[i-1, j-1]
print("a_11 =", A[0, 0], "->", names[0], "rated", movies[0])
print("a_32 =", A[2, 1], "->", names[2], "rated", movies[1])
print("a_21 =", A[1, 0], "->", names[1], "rated", movies[0])
print()

print("Expected shapes of the factors:")
print(f"  A      is {m} x {n}")
print(f"  U      is {m} x {m}   (about the PEOPLE)")
print(f"  Sigma  is {m} x {n}")
print(f"  V^T    is {n} x {n}   (about the MOVIES)")

STEP 0 - the data matrix A
A =
[[5. 2.]
 [4. 4.]
 [2. 5.]]

shape (m x n): (3, 2)
m = 3 people, n = 2 movies

a_11 = 5.0 -> John rated Movie 1
a_32 = 5.0 -> David rated Movie 2
a_21 = 4.0 -> Mary rated Movie 1

Expected shapes of the factors:
  A      is 3 x 2
  U      is 3 x 3   (about the PEOPLE)
  Sigma  is 3 x 2
  V^T    is 2 x 2   (about the MOVIES)


---
## Step 1 — The transpose $A^{\mathsf T}$

The **transpose** turns rows into columns: $(A^{\mathsf T})_{ij} = a_{ji}$.
If $A$ is $m\times n$ then $A^{\mathsf T}$ is $n\times m$ — the dimensions flip.

In [3]:
rule("STEP 1 - transpose")

At = A.T

show("A", A)
show("A^T", At)

print("shape of A  :", A.shape)
print("shape of A^T:", At.shape, " <- flipped")
print()
print("check (A^T)[0,2] == A[2,0]:", At[0, 2], "==", A[2, 0],
      "->", At[0, 2] == A[2, 0])

STEP 1 - transpose
A =
[[5. 2.]
 [4. 4.]
 [2. 5.]]

A^T =
[[5. 4. 2.]
 [2. 4. 5.]]

shape of A  : (3, 2)
shape of A^T: (2, 3)  <- flipped

check (A^T)[0,2] == A[2,0]: 2.0 == 2.0 -> True


---
## Step 2 — The products $A^{\mathsf T}A$ and $AA^{\mathsf T}$

Matrix multiplication is legal only when the **inner dimensions match**, and each
entry is a dot product:

$$ c_{ik} = (\text{row } i \text{ of the first}) \cdot (\text{column } k \text{ of the second}) $$

- $A^{\mathsf T}A$ is $2\times2$ — it compares **movies** with each other.
- $AA^{\mathsf T}$ is $3\times3$ — it compares **people** with each other.

Both are always **symmetric**, which is what guarantees real eigenvalues in Step 3.

In [4]:
rule("STEP 2a - A^T A  (2x2, the 'movie' matrix)")

AtA = A.T @ A          # (2x3)(3x2) -> 2x2
show("A^T A", AtA)

# reproduce two entries by hand, as dot products of the COLUMNS of A
col1, col2 = A[:, 0], A[:, 1]
print("entry (1,1) by hand = col1 . col1 =", col1 @ col1)
print("entry (1,2) by hand = col1 . col2 =", col1 @ col2)
print()
print("symmetric?", np.allclose(AtA, AtA.T))
print()
print("Meaning: the off-diagonal entry is how much the two movies'")
print("rating patterns agree across all three people.")

STEP 2a - A^T A  (2x2, the 'movie' matrix)
A^T A =
[[45. 36.]
 [36. 45.]]

entry (1,1) by hand = col1 . col1 = 45.0
entry (1,2) by hand = col1 . col2 = 36.0

symmetric? True

Meaning: the off-diagonal entry is how much the two movies'
rating patterns agree across all three people.


In [5]:
rule("STEP 2b - A A^T  (3x3, the 'people' matrix)")

AAt = A @ A.T          # (3x2)(2x3) -> 3x3
show("A A^T", AAt)

print("symmetric?", np.allclose(AAt, AAt.T))
print()
print("Off-diagonal entries = similarity between two people:")
for i in range(3):
    for j in range(i + 1, 3):
        print(f"  {names[i]:<6} . {names[j]:<6} = {AAt[i, j]:6.1f}")
print()
lo = min(((AAt[i, j], i, j) for i in range(3) for j in range(i + 1, 3)))
print(f"Smallest -> {names[lo[1]]} and {names[lo[2]]} are the least alike.")

STEP 2b - A A^T  (3x3, the 'people' matrix)
A A^T =
[[29. 28. 20.]
 [28. 32. 28.]
 [20. 28. 29.]]

symmetric? True

Off-diagonal entries = similarity between two people:
  John   . Mary   =   28.0
  John   . David  =   20.0
  Mary   . David  =   28.0

Smallest -> John and David are the least alike.


---
## Step 3 — Eigenvalues

An **eigenvector** $v$ satisfies $Mv = \lambda v$ — multiplying by $M$ changes only
its length, not its direction. The $\lambda$ solve the **characteristic equation**

$$ \det(M - \lambda I) = 0 $$

For a $2\times2$ matrix this is a quadratic, and we can solve it directly from the
**trace** and **determinant** rather than expanding symbolically:

$$ \lambda^2 - (\text{trace})\lambda + \det = 0
\quad\Longrightarrow\quad
\lambda = \frac{\text{trace} \pm \sqrt{\text{trace}^2 - 4\det}}{2} $$

In [ ]:
rule("STEP 3 - eigenvalues of A^T A, computed from scratch")

tr  = AtA[0, 0] + AtA[1, 1]                      # trace
det = AtA[0, 0] * AtA[1, 1] - AtA[0, 1] * AtA[1, 0]   # ad - bc
disc = tr**2 - 4 * det                            # discriminant

print(f"trace         = {tr}")
print(f"determinant   = {det}")
print(f"discriminant  = {disc}   (sqrt = {np.sqrt(disc)})")
print()

lam1 = (tr + np.sqrt(disc)) / 2
lam2 = (tr - np.sqrt(disc)) / 2
eigenvalues = np.array([lam1, lam2])      # already descending

print(f"lambda_1 = {lam1}")
print(f"lambda_2 = {lam2}")
print()

# two independent self-checks
print("CHECK  sum  :", lam1 + lam2, "== trace", tr, "->", np.isclose(lam1 + lam2, tr))
print("CHECK  prod :", lam1 * lam2, "== det  ", det, "->", np.isclose(lam1 * lam2, det))

In [ ]:
rule("STEP 3 - cross-check with numpy, and the 3x3 route")

# numpy: eigvalsh is for SYMMETRIC matrices; it returns ASCENDING order
print("np.linalg.eigvalsh(A^T A) descending:",
      np.sort(np.linalg.eigvalsh(AtA))[::-1])
print("our hand values                    :", eigenvalues)
print()

print("np.linalg.eigvalsh(A A^T) descending:",
      np.sort(np.linalg.eigvalsh(AAt))[::-1])
print()
print("The 3x3 matrix has the SAME nonzero eigenvalues plus one extra 0.")
print("The count of nonzero eigenvalues is the RANK:")
print("  rank(A) =", np.linalg.matrix_rank(A))
print()
print("That 0 is expected: three people, but only two movies, so the")
print("three rating vectors must lie in a 2-dimensional space.")
print()
print("NOTE: the printed 0 may show as -0. or 1e-16 -- floating point never")
print("      lands exactly on zero, which is why rank uses a tolerance.")

---
## Step 4 — Singular values and $\Sigma$

$$ \sigma_i = \sqrt{\lambda_i}, \qquad \sigma_1 \ge \sigma_2 \ge \dots \ge 0 $$

$\Sigma$ must be the **same shape as $A$** for the full SVD ($3\times2$ here), so the
leftover row is padded with zeros. The *reduced* form keeps just the $2\times2$ block.

The share of the structure carried by pattern $i$ is $\sigma_i^2 \big/ \sum_j \sigma_j^2$.

In [ ]:
rule("STEP 4 - singular values and Sigma")

sigma = np.sqrt(eigenvalues)
s1, s2 = sigma

print(f"sigma_1 = sqrt({lam1}) = {s1}")
print(f"sigma_2 = sqrt({lam2}) = {s2}")
print()

Sigma_full = np.zeros_like(A)          # 3x2, same shape as A
np.fill_diagonal(Sigma_full, sigma)
Sigma_red = np.diag(sigma)             # 2x2

show("Sigma (full, 3x2)", Sigma_full)
show("Sigma (reduced, 2x2)", Sigma_red)

energy = sigma**2 / np.sum(sigma**2)
print("Importance of each pattern:")
for i, e in enumerate(energy, start=1):
    print(f"  pattern {i}: sigma_{i}^2 / total = {e:.4f}  ->  {100*e:5.1f}%")
print()
print("CHECK  percentages sum to 1:", np.isclose(energy.sum(), 1.0))

---
## Step 5 — Eigenvectors of $A^{\mathsf T}A$ give $V$

For each $\lambda$, solve $(A^{\mathsf T}A - \lambda I)v = 0$.

Because the determinant is zero the two equations are **redundant** (one is a multiple
of the other), so there are infinitely many solutions along a line. Any nonzero vector
on that line works.

**The trick for a singular $2\times2$:** if a row is $[a,\ b]$, then $[-b,\ a]$ is
orthogonal to it, and therefore lies exactly in the null space we want.

Finally **normalise** each vector to length 1 and stack them as the **columns** of $V$,
ordered by descending eigenvalue.

In [ ]:
rule("STEP 5 - eigenvectors -> V")

def null_vector_2x2(M):
    """Nonzero v with M @ v = 0, for a singular 2x2 M.
    If a row is [a, b], then [-b, a] is perpendicular to it."""
    a, b = M[0]
    v = np.array([-b, a])
    if np.allclose(v, 0):            # first row was all zeros
        a, b = M[1]
        v = np.array([-b, a])
    return v


def tidy_sign(v):
    """Cosmetic only: make the first nonzero entry positive, so the output
    matches the by-hand answer. Flipping a sign gives an equally valid vector."""
    first = v[np.argmax(np.abs(v) > 1e-12)]
    return -v if first < 0 else v


def simplify_int(v):
    """If v is all (near) integers, divide out the GCD to get the smallest
    equivalent vector -- i.e. [36, 36] becomes [1, 1], which is what you
    would pick by hand. Direction is unchanged, so it is still the same
    eigenvector."""
    r = np.rint(v)
    if not np.allclose(v, r):
        return v
    r = r.astype(int)
    g = np.gcd.reduce(np.abs(r))
    return r // g if g > 0 else r


I2 = np.eye(2)

for lam, tag in [(lam1, "lambda_1"), (lam2, "lambda_2")]:
    print(f"--- {tag} = {lam} ---")
    show(f"A^T A - ({lam}) I", AtA - lam * I2)

v1 = simplify_int(tidy_sign(null_vector_2x2(AtA - lam1 * I2)))
v2 = simplify_int(tidy_sign(null_vector_2x2(AtA - lam2 * I2)))

print("eigenvector for lambda_1:", v1, " (any nonzero multiple is valid)")
print("eigenvector for lambda_2:", v2)
print()

# verify they really are eigenvectors before normalising
print("CHECK (A^T A - lam1 I) v1 = 0 ->", np.allclose((AtA - lam1 * I2) @ v1, 0))
print("CHECK (A^T A - lam2 I) v2 = 0 ->", np.allclose((AtA - lam2 * I2) @ v2, 0))

In [ ]:
rule("STEP 5 - normalise and assemble V")

v1_hat = v1 / np.linalg.norm(v1)
v2_hat = v2 / np.linalg.norm(v2)

print("v1 normalised:", np.round(v1_hat, 4), "   (= [1, 1]/sqrt(2))")
print("v2 normalised:", np.round(v2_hat, 4), "   (= [1, -1]/sqrt(2))")
print("1/sqrt(2) =", round(1/np.sqrt(2), 4))
print()

V  = np.column_stack([v1_hat, v2_hat])   # eigenvectors are COLUMNS
Vt = V.T

show("V", V)
show("V^T", Vt)

print("CHECK  lengths are 1 :", np.isclose(np.linalg.norm(v1_hat), 1),
      np.isclose(np.linalg.norm(v2_hat), 1))
print("CHECK  orthogonal    : v1 . v2 =", round(float(v1_hat @ v2_hat), 12))
print("CHECK  V^T V = I     :", np.allclose(V.T @ V, np.eye(2)))

---
## Step 6 — Build $U$ from the linking formula

Left and right singular vectors are tied together by

$$ A v_i = \sigma_i u_i \qquad\Longleftrightarrow\qquad u_i = \frac{1}{\sigma_i} A v_i $$

So each column of $U$ costs one matrix–vector product — **no second
eigendecomposition needed**. The result is automatically a unit vector.

This only works when $\sigma_i \neq 0$. Since $\sigma_3 = 0$, the third column of $U$
must come from somewhere else: we need any unit vector perpendicular to both $u_1$ and
$u_2$, and in $\mathbb{R}^3$ the **cross product** gives exactly that.

In [ ]:
rule("STEP 6 - U from u_i = A v_i / sigma_i")

u1 = A @ v1_hat / s1
u2 = A @ v2_hat / s2

print("A @ v1 =", np.round(A @ v1_hat, 4), " then / sigma_1 =", s1)
print("u1     =", np.round(u1, 4))
print()
print("A @ v2 =", np.round(A @ v2_hat, 4), " then / sigma_2 =", s2)
print("u2     =", np.round(u2, 4))
print()

print("CHECK  |u1| = 1 :", np.isclose(np.linalg.norm(u1), 1))
print("CHECK  |u2| = 1 :", np.isclose(np.linalg.norm(u2), 1))
print("CHECK  u1 . u2  =", round(float(u1 @ u2), 12))

In [ ]:
rule("STEP 6c - third column via the cross product")

u3 = np.cross(u1, u2)
print("cross(u1, u2)      =", np.round(u3, 4))
u3 = u3 / np.linalg.norm(u3)
print("normalised u3      =", np.round(u3, 4), "  (= [-4, 7, -4]/9)")
print()

print("CHECK  u3 . u1 =", round(float(u3 @ u1), 12))
print("CHECK  u3 . u2 =", round(float(u3 @ u2), 12))
print("CHECK  |u3| = 1:", np.isclose(np.linalg.norm(u3), 1))
print()

U_full = np.column_stack([u1, u2, u3])   # 3x3
U_red  = np.column_stack([u1, u2])       # 3x2

show("U (full, 3x3)", U_full)
show("U (reduced, 3x2)", U_red)

print("CHECK  U is orthogonal (U^T U = I):",
      np.allclose(U_full.T @ U_full, np.eye(3)))

---
## Step 7 — Assemble and **verify**

This is the step that proves the whole thing. If $U\Sigma V^{\mathsf T}$ does not come
back to $A$, something earlier is wrong — almost always eigenvalues out of order, a
vector left un-normalised, or eigenvectors written as rows instead of columns.

The full and reduced forms give the *same* product, because the discarded column of $U$
only ever multiplies the zero row of $\Sigma$.

In [ ]:
rule("STEP 7 - assemble and verify")

recon_full = U_full @ Sigma_full @ Vt
recon_red  = U_red  @ Sigma_red  @ Vt

show("U @ Sigma @ V^T   (full)", recon_full)
show("U @ Sigma @ V^T   (reduced)", recon_red)
show("original A", A)

print("full form    reproduces A :", np.allclose(recon_full, A))
print("reduced form reproduces A :", np.allclose(recon_red, A))
print("max abs difference (full) :", np.abs(recon_full - A).max())
print()

if np.allclose(recon_full, A):
    print(">>> SUCCESS: A = U Sigma V^T  <<<")
else:
    print(">>> MISMATCH - check ordering, normalisation, rows vs columns <<<")

---
## Step 8 — What do the numbers mean?

The SVD can be rewritten as a sum of **patterns**, strongest first:

$$ A = \sigma_1 u_1 v_1^{\mathsf T} + \sigma_2 u_2 v_2^{\mathsf T} $$

Each $u_i$ describes the **people**, each $v_i$ describes the **movies**, and $\sigma_i$
says how strong that pattern is. Read the **signs**: entries with the same sign go
together, opposite signs are in tension, and a zero means *no involvement at all*.

In [ ]:
rule("STEP 8 - reading the two patterns")

for k, (u, v, sg) in enumerate([(u1, v1_hat, s1), (u2, v2_hat, s2)], start=1):
    print(f"--- PATTERN {k}   sigma_{k} = {sg:.4f}   "
          f"({100*energy[k-1]:.1f}% of the structure) ---")
    print("  people (u{}):".format(k))
    for nm, val in zip(names, u):
        print(f"     {nm:<6} {val:+7.4f}")
    print("  movies (v{}):".format(k))
    for mv, val in zip(movies, v):
        print(f"     {mv:<8} {val:+7.4f}")
    same = "all the same sign -> a common level, no contrast" \
        if np.all(u > 0) or np.all(u < 0) else \
        "signs DISAGREE -> this pattern is a contrast"
    print("  reading:", same)
    print()

print("Pattern 1: every entry positive - overall generosity / general popularity.")
print("Pattern 2: a contrast - one person leans to Movie 1, another to Movie 2,")
print("           and the person sitting at 0.0 has no preference at all.")
print()
print("Note Mary's entry in pattern 2 is exactly 0, which matches her ratings (4, 4).")

---
## Step 9 — Rank-1 approximation

Keep only the strongest pattern: $A_1 = \sigma_1 u_1 v_1^{\mathsf T}$.

Note $u_1 v_1^{\mathsf T}$ is an **outer** product — a column times a row, giving a full
$3\times2$ matrix (use `np.outer`, not `@`).

The **Eckart–Young theorem** says this is the *best possible* rank-1 approximation, and
predicts the error exactly:

$$ \|A - A_1\|_F = \sqrt{\sigma_2^2 + \sigma_3^2 + \cdots} = \sigma_2 $$

In [ ]:
rule("STEP 9 - rank-1 approximation")

A1 = s1 * np.outer(u1, v1_hat)

show("A_1 = sigma_1 * u1 v1^T", A1)
show("original A", A)

err_matrix = A - A1
show("A - A_1  (what we threw away)", err_matrix)

frob = np.linalg.norm(err_matrix)      # Frobenius norm by default
print(f"||A - A_1||_F = {frob}")
print(f"sigma_2       = {s2}")
print("Eckart-Young says these must be equal ->", np.isclose(frob, s2))
print()

for nm, row in zip(names, err_matrix):
    tag = "  <- exactly zero: no taste component to lose" if np.allclose(row, 0) else ""
    print(f"  {nm:<6} error {np.round(row, 3)}{tag}")

---
## Step 10 — The pseudoinverse

A non-square matrix has no inverse, but the SVD gives the next best thing:

$$ A^{+} = V \Sigma^{-1} U^{\mathsf T},
\qquad \Sigma^{-1} = \operatorname{diag}\!\left(\tfrac{1}{\sigma_1}, \dots, \tfrac{1}{\sigma_r}\right) $$

Invert only the **nonzero** singular values — zeros stay zero, never $1/0$.

In [ ]:
rule("STEP 10 - pseudoinverse")

Sigma_inv = np.diag(1.0 / sigma)        # only the nonzero sigmas
A_plus = V @ Sigma_inv @ U_red.T

show("A^+ = V Sigma^-1 U^T", A_plus)

print("matches np.linalg.pinv(A):", np.allclose(A_plus, np.linalg.pinv(A)))
print()
show("A^+ * 81  (whole numbers!)", A_plus * 81)
print("so A^+ = (1/81) * [[17, 4, -10], [-10, 4, 17]]")
print()
print("CHECK  A^+ A = I (2x2), since A has full COLUMN rank:")
show("A^+ @ A", A_plus @ A)

---
## Step 11 — Cross-check against `np.linalg.svd`

Everything above was built from scratch. Now compare with NumPy's own routine.

`np.linalg.svd` returns `s` as a **1-D array**, not a matrix — rebuild with
`np.diag(s)`.

**Expect the signs to differ.** If $(u_i, v_i)$ is a valid pair then so is
$(-u_i, -v_i)$, because the two minus signs cancel in $\sigma_i u_i v_i^{\mathsf T}$.
Each column may flip independently. That is *not* an error — which is exactly why you
compare the **product**, never the raw signs.

In [ ]:
rule("STEP 11 - compare with np.linalg.svd")

U_np, s_np, Vt_np = np.linalg.svd(A, full_matrices=False)

print("singular values")
print("  ours :", np.round(sigma, 6))
print("  numpy:", np.round(s_np, 6))
print("  match:", np.allclose(sigma, s_np))
print()

show("our U (reduced)", U_red)
show("numpy U", U_np)
show("our V^T", Vt)
show("numpy V^T", Vt_np)

print("Column-by-column comparison:")
for i in range(2):
    flipped = np.allclose(U_red[:, i], -U_np[:, i])
    same    = np.allclose(U_red[:, i], U_np[:, i])
    print(f"  column {i+1}: identical={same}  sign-flipped={flipped}")
print()

print("Both reconstruct A:")
print("  ours :", np.allclose(U_red @ np.diag(sigma) @ Vt, A))
print("  numpy:", np.allclose(U_np @ np.diag(s_np) @ Vt_np, A))
print()
print("Same subspaces either way -- |dot product| between matching columns:")
for i in range(2):
    print(f"  |u{i+1} . u{i+1}_numpy| =", round(abs(float(U_red[:, i] @ U_np[:, i])), 10))

---
## Summary

Everything in one place.

In [ ]:
rule("SUMMARY - full SVD of the movie ratings matrix")

print(f"{'Quantity':<26} Value")
print("-" * 62)
print(f"{'A':<26} {A.shape[0]} x {A.shape[1]}  (people x movies)")
print(f"{'A^T A':<26} {np.round(AtA,1).tolist()}")
print(f"{'eigenvalues':<26} {np.round(eigenvalues,4).tolist()}")
print(f"{'singular values':<26} {np.round(sigma,4).tolist()}")
print(f"{'rank(A)':<26} {np.linalg.matrix_rank(A)}")
print(f"{'importance split':<26} {[f'{100*e:.0f}%' for e in energy]}")
print(f"{'||A - A_1||_F':<26} {np.linalg.norm(A - A1):.4f}  (= sigma_2)")
print(f"{'A = U Sigma V^T ?':<26} {np.allclose(U_full @ Sigma_full @ Vt, A)}")
print("-" * 62)
print()

show("U (full)", U_full)
show("Sigma (full)", Sigma_full)
show("V^T", Vt)

print("Reconstruction check:")
show("U @ Sigma @ V^T", U_full @ Sigma_full @ Vt)
print("equals A:", np.allclose(U_full @ Sigma_full @ Vt, A))

---
## Try it yourself

1. **Change one rating** — set Mary's Movie 2 rating to `5` and re-run every cell.
   Do the singular values stay whole numbers? Why were the numbers in this example so
   unusually tidy?
2. **Make the two patterns equally important** — what would the ratings have to look
   like for the split to be 50/50 instead of 90/10?
3. **Add a fourth person.** $A$ becomes $4\times2$. Which factor changes shape, $U$ or
   $V$? Predict first, then check.
4. **Break it on purpose** — swap the order of the two eigenvalues (use `lam2` where
   `lam1` belongs) and watch Step 7 fail. This is the single most common mistake.